# Logistic Regression (Model -1)

In [12]:
# MODEL 1 - Logistic Regression with TF-IDF features
# TF-IDF converts text to numbers based on word importance
# Logistic Regression then classifies which answer (A-E) is correct

# ==================== creating TF-IDF features =======================

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf    = tfidf_vectorizer.fit_transform(train['full_text'])
X_test_tfidf     = tfidf_vectorizer.transform(test['full_text'])

print('TF-IDF feature matrix shape:', X_train_tfidf.shape)

# ============ starting wandb run for logistic regression ==================

run1 = wandb.init(
    project = '23f3001514-t22026',
    entity  = '23f3001514-instituition',
    name    = 'logistic-regression-tfidf',
    config  = {
        'model'        : 'LogisticRegression',
        'features'     : 'TF-IDF',
        'max_features' : 5000,
        'C'            : 1.0,
        'max_iter'     : 1000
    }
)

# ============== training logistic regression =================

lr_model = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', multi_class='multinomial')
lr_model.fit(X_train_tfidf, y_train)

# ============ calculating cross validation scores ==================

lr_f1_scores  = cross_val_score(lr_model, X_train_tfidf, y_train, cv=5, scoring='f1_macro')
lr_acc_scores = cross_val_score(lr_model, X_train_tfidf, y_train, cv=5, scoring='accuracy')

# =========== calculating mean score of metrics ===============

lr_f1  = lr_f1_scores.mean()
lr_acc = lr_acc_scores.mean()

# ============= calculating mAP@3 on training set ===============

lr_train_probs = lr_model.predict_proba(X_train_tfidf)
lr_train_top3  = []
for i in range(len(lr_train_probs)):
    top3 = get_top3_from_probs(lr_train_probs[i], list(lr_model.classes_))
    lr_train_top3.append(top3)

lr_map3 = map_at_k(y_train.tolist(), lr_train_top3)

print('Logistic Regression Results:')
print('  CV Macro F1  :', round(lr_f1, 4))
print('  CV Accuracy  :', round(lr_acc, 4))
print('  Train mAP@3  :', round(lr_map3, 4))

# =========== logging to wandb ==================

wandb.log({
    'macro_f1' : lr_f1,
    'accuracy' : lr_acc,
    'map_at_3' : lr_map3
})
run1.finish()

# =========== getting test predictions =================

lr_test_probs = lr_model.predict_proba(X_test_tfidf)
print('='*50)
print('LR done!')

TF-IDF feature matrix shape: (2000, 2940)


Logistic Regression Results:
  CV Macro F1  : 0.9987
  CV Accuracy  : 0.9985
  Train mAP@3  : 1.0


accuracy,▁
macro_f1,▁
map_at_3,▁
accuracy,0.9985
macro_f1,0.99873
map_at_3,1


LR done!


**INSIGHT:**

Logistic Regression Insights:

1. TF-IDF created only 2940 features out of max 5000.
   This means the dataset vocabulary is small and domain specific.

2. CV Accuracy of 99.85% and Macro F1 of 99.87% seem very high.
   This suggests the model is learning a strong lexical overlap pattern —
   the correct answer tends to repeat key words from the question prompt.
   TF-IDF captures this overlap very effectively.

3. Train mAP@3 of 1.0 means for every question the correct answer
   appeared in the top 3 predictions. Perfect ranking on training data.


**INSIGHT:**

1. Vocabulary size is only 2975 words — much smaller than our
   VOCAB_SIZE_MAX of 15000. This confirms the dataset is domain
   specific with limited unique terminology across all questions.

2. MAX_LENGTH of 256 was chosen based on EDA — most questions
   with all 5 options combined are under 256 words. Setting it
   higher would waste memory with unnecessary padding.

3. Words appearing only once are excluded (count >= 2).
   Single occurrence words are likely typos or rare terms
   that don't help the model learn general patterns.

4. Train/val split of 80/20 gives 1600 training samples and
   400 validation samples. Validation data is never used for
   training — only for checking generalization during training.

5. Batch size of 32 is a good balance between memory efficiency
   and training stability. Too large = GPU memory error.
   Too small = noisy gradient updates.

# TEXTCNN (Model -2)

In [21]:
# MODEL 2 - TextCNN (built completely from scratch)
# CNN for text classification works by sliding filters over the word embeddings
# different filter sizes capture different length patterns (bigrams, trigrams etc)
# i am using kernel sizes 2, 3, 4 to capture 2-word, 3-word and 4-word patterns

class TextCNN(nn.Module):

    def __init__(self, vocab_size, embedding_dim, num_classes, num_filters, kernel_sizes):
        super(TextCNN, self).__init__()

        # ======= embedding layer converts word indices to dense vectors =========
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        #  ========== convolutional layers with different kernel sizes ===========
        # ========== each one captures patterns of different lengths ============
        
        self.conv2 = nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=2)
        self.conv3 = nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=3)
        self.conv4 = nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=4)

        # ====== dropout for regularization - prevents overfitting ========
        
        self.dropout = nn.Dropout(p=0.5)

        #  ====== final linear layer : maps to number of classes (5 options) ===========
        #  =========== 3 * num_filters because we have 3 different kernel sizes =============
        
        self.fc = nn.Linear(3 * num_filters, num_classes)

    def forward(self, x):
        # x shape: (batch_size, sequence_length)

        # ======= getting embeddings ========
        
        embedded = self.embedding(x)
        # ===== shape: (batch_size, sequence_length, embedding_dim) =========

        # ======= conv1d expects (batch_size, channels, length) so we permute the dimensions ==========
     
        embedded = embedded.permute(0, 2, 1) # shape: (batch_size, embedding_dim, sequence_length)

        # ========== applying each convolutional filter and relu activation ===============
        
        out2 = torch.relu(self.conv2(embedded))
        out3 = torch.relu(self.conv3(embedded))
        out4 = torch.relu(self.conv4(embedded))

        # ========= global max pooling : keeps only the highest value from each filter =============
        
        pool2 = torch.max(out2, dim=2)[0]
        pool3 = torch.max(out3, dim=2)[0]
        pool4 = torch.max(out4, dim=2)[0]

        # ======= concatenating all pooled outputs in a single long vector ===========
        
        concatenated = torch.cat([pool2, pool3, pool4], dim=1)

        # ===== applying dropout =======
        
        dropped = self.dropout(concatenated)

        #  ======= final classification =======
        
        output = self.fc(dropped)

        return output


# ================ creating the TextCNN model ====================

cnn_model = TextCNN(
    vocab_size    = VOCAB_SIZE,
    embedding_dim = 128,
    num_classes   = 5,
    num_filters   = 128,
    kernel_sizes  = [2, 3, 4]
)
cnn_model = cnn_model.to(device)

total_params = sum(p.numel() for p in cnn_model.parameters())
print('TextCNN total parameters:', total_params)
print('='*50)
print()
print(cnn_model)

TextCNN total parameters: 530565

TextCNN(
  (embedding): Embedding(2975, 128, padding_idx=0)
  (conv2): Conv1d(128, 128, kernel_size=(2,), stride=(1,))
  (conv3): Conv1d(128, 128, kernel_size=(3,), stride=(1,))
  (conv4): Conv1d(128, 128, kernel_size=(4,), stride=(1,))
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=384, out_features=5, bias=True)
)


**INSIGHT :**

1. Total parameters: 530,565
   Most parameters are in the embedding layer (380,800)
   which learns word representations during training.

2. Three separate conv filters capture patterns of different lengths.
   kernel_size=2 catches short patterns like "not correct"
   kernel_size=3 catches medium patterns like "is the answer"
   kernel_size=4 catches longer contextual patterns.

3. Global max pooling reduces each filter output to a single value
   — the most important feature found anywhere in the text.
   This makes the model position-invariant — it doesn't matter
   where in the text the pattern appears.

4. Dropout of 0.5 randomly disables half the neurons during training.
   This forces the model to learn redundant representations
   and prevents it from memorizing training examples.

5. Final layer maps 384 features → 5 scores.
   The class with highest score = predicted answer.

# TEXTCRNN (Model -3)

In [23]:
# MODEL 3 - TextCRNN (CNN + Bidirectional LSTM) built from scratch
# this model combines CNN for local patterns and LSTM for sequential patterns
# bidirectional LSTM reads the text both forward and backward
# this helps understand context from both directions

class TextCRNN(nn.Module):

    def __init__(self, vocab_size, embedding_dim, num_classes, num_filters, hidden_size):
        super(TextCRNN, self).__init__()

        # ==== embedding layer =====
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # ======= CNN layer to extract local features =======
        self.conv = nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=3, padding=1)

        # ========== bidirectional LSTM with 2 layers =============
        # ====== bidirectional means it processes text both left to right and right to left ===========
        
        self.lstm = nn.LSTM(
            input_size    = num_filters,
            hidden_size   = hidden_size,
            num_layers    = 2,
            batch_first   = True,
            dropout       = 0.3,
            bidirectional = True
        )

        self.dropout = nn.Dropout(p=0.5)

        # ========== hidden_size * 2 because bidirectional (forward + backward) ============
        
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        # x shape: (batch_size, sequence_length)

        # embedding
        embedded = self.embedding(x)
        # shape: (batch_size, sequence_length, embedding_dim)

        # permute for conv1d
        embedded = embedded.permute(0, 2, 1) # shape: (batch_size, embedding_dim, sequence_length)

        # CNN to get local features
        conv_out = torch.relu(self.conv(embedded))
        # shape: (batch_size, num_filters, sequence_length)

        # ===== permute back for LSTM =====
        conv_out = conv_out.permute(0, 2, 1) # shape: (batch_size, sequence_length, num_filters)

        # ======= LSTM : we only need the final hidden state ==========
        lstm_out, (hidden, cell) = self.lstm(conv_out)

        # ====== hidden shape: (num_layers * 2, batch_size, hidden_size) ==========
        # ====== taking last layer forward and backward hidden states ============
        
        forward_hidden  = hidden[-2]  # ===== last layer forward =======
        backward_hidden = hidden[-1]  # ====== last layer backward ========

        # ====== concatenating both directions ========
        
        combined = torch.cat([forward_hidden, backward_hidden], dim=1) # shape: (batch_size, hidden_size * 2)

        dropped = self.dropout(combined)
        output  = self.fc(dropped)

        return output


# =============== creating the TextCRNN model ===============

crnn_model = TextCRNN(
    vocab_size    = VOCAB_SIZE,
    embedding_dim = 128,
    num_classes   = 5,
    num_filters   = 128,
    hidden_size   = 128
)
crnn_model = crnn_model.to(device)

total_params = sum(p.numel() for p in crnn_model.parameters())
print('TextCRNN total parameters:', total_params)
print('='*50)
print()
print(crnn_model)

TextCRNN total parameters: 1090821

TextCRNN(
  (embedding): Embedding(2975, 128, padding_idx=0)
  (conv): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=5, bias=True)
)


**INSIGHT:**

1. Total parameters: 1,090,821
   More than double TextCNN (530,565) because BiLSTM adds
   significant complexity with its gating mechanisms.

2. CNN + BiLSTM combination gives two types of understanding:
   CNN → local patterns (what words appear together)
   BiLSTM → sequential context (how meaning flows through text)

3. padding=1 in conv layer keeps sequence length unchanged.
   This is necessary because LSTM needs the complete sequence
   to understand long range dependencies.

4. bidirectional=True means two LSTM passes happen:
   Forward: reads question left to right
   Backward: reads question right to left
   Combining both gives the model full sentence context.

5. num_layers=2 means two LSTM layers are stacked.
   First layer learns basic patterns.
   Second layer learns more abstract higher level patterns.

6. Despite having 2x more parameters than TextCNN,
   TextCRNN received only 10% weight in ensemble.
   This is because with only 2000 training samples,
   the larger model did not significantly outperform TextCNN.
   More data would likely benefit TextCRNN more.

# SBERT (Pretrained Model)

In [25]:
# MODEL 4 - Pretrained Model from HuggingFace
# ======== model name: all-MiniLM-L6-v2 ===========
# ========= this is a BERT based model that was already trained on millions of sentences =============
# ======== it converts any sentence into a list of numbers (called embeddings) ============
# ======== I use these numbers to find which answer option is most similar to the question ==========

import subprocess
subprocess.run(['pip', 'install', 'sentence-transformers', '-q'])

from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score

# ========== loading the pretrained model =============
# ========== this model was trained by HuggingFace on 1 billion sentence pairs =============
# ======== we are NOT training it - just using it directly =============

print('loading pretrained model...')
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
print('model loaded!')


def get_sbert_predictions(df, model):
    # ====== for each question we will: ========
    # 1. convert the prompt to numbers (embedding)
    # 2. convert each option A B C D E to numbers (embedding)
    # 3. compare prompt with each option using cosine similarity
    # 4. option with highest similarity = most likely answer

    options = ['A', 'B', 'C', 'D', 'E']

    # ===== creating empty array to store similarity scores =========
    # ==== shape is (number of questions, 5 options) =======
    
    similarity_scores = np.zeros((len(df), 5))

    # ========== step 1 : convert all prompts to number vectors =============
    
    print('converting prompts to embeddings...')
    prompt_embeddings = model.encode(
        df['prompt'].tolist(),
        batch_size           = 64,
        show_progress_bar    = True,
        normalize_embeddings = True   # makes cosine similarity = dot product
    )

    # ========= step 2 : convert each option to number vectors ==============
    # ========== step 3 : compare with prompt using dot product (cosine similarity) =============
    
    for i in range(len(options)):
        opt = options[i]
        print('converting option', opt, 'to embeddings...')

        option_embeddings = model.encode(
            df[opt].tolist(),
            batch_size           = 64,
            show_progress_bar    = False,
            normalize_embeddings = True
        )

        # dot product of two normalized vectors = cosine similarity
        # higher value means more similar
        scores = (prompt_embeddings * option_embeddings).sum(axis=1)
        similarity_scores[:, i] = scores

    # step 4 - convert similarity scores to probabilities using softmax
    # subtract max for numerical stability (prevents overflow)
    similarity_scores = similarity_scores - similarity_scores.max(axis=1, keepdims=True)
    similarity_scores = np.exp(similarity_scores)
    similarity_scores = similarity_scores / similarity_scores.sum(axis=1, keepdims=True)

    return similarity_scores


# ========== getting predictions on training data to evaluate performance ===========

print('running pretrained model on training data...')
sbert_train_probs = get_sbert_predictions(train, sbert_model)

# ========= getting top 3 predictions for each question =============

sbert_train_top3 = []
for i in range(len(sbert_train_probs)):
    top3 = get_top3_from_probs(sbert_train_probs[i], ['A', 'B', 'C', 'D', 'E'])
    sbert_train_top3.append(top3)

# ======== calculating evaluation metrics ==========

sbert_map3 = map_at_k(y_train.tolist(), sbert_train_top3)
sbert_f1   = f1_score(y_train, [p[0] for p in sbert_train_top3], average='macro')
sbert_acc  = accuracy_score(y_train, [p[0] for p in sbert_train_top3])

print('pretrained model results on training data:')
print('  mAP@3    :', round(sbert_map3, 4))
print('  Macro F1 :', round(sbert_f1, 4))
print('  Accuracy :', round(sbert_acc, 4))


# ============= logging pretrained model to wandb ==============

run_sbert = wandb.init(
    project = '23f3001514-t22026',
    entity  = '23f3001514-instituition',
    name    = 'sbert-all-minilm-l6-v2',
    config  = {
        'model'      : 'all-MiniLM-L6-v2',
        'model_type' : 'pretrained transformer',
        'strategy'   : 'cosine similarity between prompt and options',
        'batch_size' : 64
    }
)

wandb.log({
    'macro_f1' : sbert_f1,
    'accuracy' : sbert_acc,
    'map_at_3' : sbert_map3
})

run_sbert.finish()
print('pretrained model results logged to wandb!')


# ========= getting predictions on test data ==============

print('running pretrained model on test data...')
sbert_test_probs = get_sbert_predictions(test, sbert_model)
print('done! shape:', sbert_test_probs.shape)

loading pretrained model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

converting option A to embeddings...
converting option B to embeddings...
converting option C to embeddings...
converting option D to embeddings...
converting option E to embeddings...
pretrained model results on training data:
  mAP@3    : 0.4231
  Macro F1 : 0.2587
  Accuracy : 0.261


accuracy,▁
macro_f1,▁
map_at_3,▁
accuracy,0.261
macro_f1,0.25865
map_at_3,0.42308


pretrained model results logged to wandb!
running pretrained model on test data...
converting prompts to embeddings...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

converting option A to embeddings...
converting option B to embeddings...
converting option C to embeddings...
converting option D to embeddings...
converting option E to embeddings...
done! shape: (500, 5)
